In [ ]:
import numpy as np
from scipy.optimize import minimize, minimize_scalar, brentq


# ------------------------------------------------------------
# PSF model
# ------------------------------------------------------------

def psf_shape(th2, psf_params):
    """
    Triple-Gaussian PSF shape.

    Parameters
    ----------
    th2 : array
        theta^2 bin centers.
    psf_params : array-like
        sigma_1, sigma_2, sigma_3, scale, ampl_2, ampl_3
    """

    sigma_1, sigma_2, sigma_3, scale, ampl_2, ampl_3 = psf_params

    gauss1 = np.exp(-th2 / (2.0 * sigma_1**2))
    gauss2 = ampl_2 * np.exp(-th2 / (2.0 * sigma_2**2))
    gauss3 = ampl_3 * np.exp(-th2 / (2.0 * sigma_3**2))

    return scale * (gauss1 + gauss2 + gauss3)


# ------------------------------------------------------------
# Extended source convolved with the PSF
# ------------------------------------------------------------

def extended_shape(th2, sigma_ext, psf_params):
    """
    PSF-convolved Gaussian extension.

    sigma_ext is fixed while calculating one upper limit.
    """

    sigma_1, sigma_2, sigma_3, scale, ampl_2, ampl_3 = psf_params

    sigma1_conv = np.sqrt(sigma_1**2 + sigma_ext**2)
    sigma2_conv = np.sqrt(sigma_2**2 + sigma_ext**2)
    sigma3_conv = np.sqrt(sigma_3**2 + sigma_ext**2)

    c1 = sigma_1 / (sigma_1**2 + sigma_ext**2)
    c2 = ampl_2 * sigma_2 / (sigma_2**2 + sigma_ext**2)
    c3 = ampl_3 * sigma_3 / (sigma_3**2 + sigma_ext**2)

    gauss1 = c1 * np.exp(-th2 / (2.0 * sigma1_conv**2))
    gauss2 = c2 * np.exp(-th2 / (2.0 * sigma2_conv**2))
    gauss3 = c3 * np.exp(-th2 / (2.0 * sigma3_conv**2))

    return gauss1 + gauss2 + gauss3


# ------------------------------------------------------------
# Template normalization
# ------------------------------------------------------------

def normalized_template(template):
    """
    Normalize a spatial template to sum to one.

    The corresponding amplitude then represents expected total counts.
    """

    template = np.asarray(template, dtype=float)

    if not np.all(np.isfinite(template)):
        raise ValueError("Template contains NaN or infinity.")

    if np.any(template < 0):
        raise ValueError("Template contains negative values.")

    total = np.sum(template)

    if total <= 0:
        raise ValueError("Template has zero total normalization.")

    return template / total


# ------------------------------------------------------------
# Poisson negative log-likelihood
# ------------------------------------------------------------

def poisson_nll(counts, expected_counts):
    """
    Poisson negative log-likelihood, omitting log(n!).
    The omitted term cancels in likelihood ratios.
    """

    counts = np.asarray(counts, dtype=float)
    expected_counts = np.asarray(expected_counts, dtype=float)

    expected_counts = np.maximum(expected_counts, 1e-12)

    return np.sum(
        expected_counts - counts * np.log(expected_counts)
    )


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

def model_counts(
    point_counts,
    ext_counts,
    point_template,
    ext_template,
    background=None,
):
    """
    Expected counts model.

    point_counts and ext_counts are total expected source counts.
    """

    model = (
        point_counts * point_template
        + ext_counts * ext_template
    )

    if background is not None:
        model = model + background

    return model


# ------------------------------------------------------------
# Fit point-only and point-plus-extension models
# ------------------------------------------------------------

def fit_point_only(
    counts,
    point_template,
    background=None,
):
    """
    Fit the point-source-only model.

    Free parameter:
        point_counts >= 0
    """

    counts = np.asarray(counts, dtype=float)

    if background is None:
        background = np.zeros_like(counts)
    else:
        background = np.asarray(background, dtype=float)

    total_counts = np.sum(counts)

    x0 = np.array([
        max(total_counts - np.sum(background), 1e-8)
    ])

    def objective(parameters):
        point_counts = parameters[0]

        expected = model_counts(
            point_counts=point_counts,
            ext_counts=0.0,
            point_template=point_template,
            ext_template=np.zeros_like(point_template),
            background=background,
        )

        return poisson_nll(counts, expected)

    result = minimize(
        objective,
        x0=x0,
        bounds=[(0.0, None)],
        method="L-BFGS-B",
    )

    if not result.success:
        raise RuntimeError(
            f"Point-only fit failed: {result.message}"
        )

    point_counts = result.x[0]

    expected = model_counts(
        point_counts=point_counts,
        ext_counts=0.0,
        point_template=point_template,
        ext_template=np.zeros_like(point_template),
        background=background,
    )

    return {
        "point_counts": point_counts,
        "expected": expected,
        "nll": poisson_nll(counts, expected),
        "result": result,
    }


def fit_point_plus_extension(
    counts,
    point_template,
    ext_template,
    background=None,
):
    """
    Fit the point-source plus fixed-width extended-source model.

    Free parameters:
        point_counts >= 0
        ext_counts >= 0
    """

    counts = np.asarray(counts, dtype=float)

    if background is None:
        background = np.zeros_like(counts)
    else:
        background = np.asarray(background, dtype=float)

    total_counts = np.sum(counts)
    source_counts = max(total_counts - np.sum(background), 1e-8)

    x0 = np.array([
        source_counts,
        0.01 * source_counts,
    ])

    def objective(parameters):
        point_counts, ext_counts = parameters

        expected = model_counts(
            point_counts=point_counts,
            ext_counts=ext_counts,
            point_template=point_template,
            ext_template=ext_template,
            background=background,
        )

        return poisson_nll(counts, expected)

    result = minimize(
        objective,
        x0=x0,
        bounds=[
            (0.0, None),
            (0.0, None),
        ],
        method="L-BFGS-B",
    )

    if not result.success:
        raise RuntimeError(
            f"Point-plus-extension fit failed: {result.message}"
        )

    point_counts, ext_counts = result.x

    expected = model_counts(
        point_counts=point_counts,
        ext_counts=ext_counts,
        point_template=point_template,
        ext_template=ext_template,
        background=background,
    )

    return {
        "point_counts": point_counts,
        "ext_counts": ext_counts,
        "expected": expected,
        "nll": poisson_nll(counts, expected),
        "result": result,
    }


# ------------------------------------------------------------
# Profile likelihood
# ------------------------------------------------------------

def profile_nll_fixed_extension(
    ext_counts,
    counts,
    point_template,
    ext_template,
    background=None,
):
    """
    Profile the likelihood at fixed extended-source counts.

    ext_counts is fixed.
    point_counts is reoptimized.
    """

    counts = np.asarray(counts, dtype=float)

    if background is None:
        background = np.zeros_like(counts)
    else:
        background = np.asarray(background, dtype=float)

    total_counts = np.sum(counts)
    source_counts = max(total_counts - np.sum(background), 1.0)

    def objective(point_parameter):
        point_counts = point_parameter[0]

        expected = model_counts(
            point_counts=point_counts,
            ext_counts=ext_counts,
            point_template=point_template,
            ext_template=ext_template,
            background=background,
        )

        return poisson_nll(counts, expected)

    result = minimize(
        objective,
        x0=[source_counts],
        bounds=[(0.0, None)],
        method="L-BFGS-B",
    )

    if not result.success:
        raise RuntimeError(
            f"Profile fit failed: {result.message}"
        )

    return result.fun, result.x[0]


def calculate_upper_limit_one_dataset(
    counts,
    th2,
    sigma_ext,
    psf_params,
    background=None,
    delta_ts=2.71,
):
    """
    Calculate one upper limit for one dataset and one extension size.

    Parameters
    ----------
    counts : array
        Observed counts in theta^2 bins.
    th2 : array
        theta^2 bin centers.
    sigma_ext : float
        Fixed extension width.
    psf_params : array
        PSF parameters.
    background : array or None
        Known background counts per bin.
    delta_ts : float
        2.71 gives an approximate one-sided 95% upper limit
        for one parameter of interest.

    Returns
    -------
    result : dict
        Contains the upper limit and fit information.
    """

    counts = np.asarray(counts, dtype=float)
    th2 = np.asarray(th2, dtype=float)

    if np.any(counts < 0):
        raise ValueError("Counts cannot be negative.")

    if background is not None:
        background = np.asarray(background, dtype=float)

        if np.any(background < 0):
            raise ValueError("Background cannot be negative.")

        if background.shape != counts.shape:
            raise ValueError(
                "background and counts must have the same shape."
            )

    # Build and normalize templates.
    point_template = normalized_template(
        psf_shape(th2, psf_params)
    )

    ext_template = normalized_template(
        extended_shape(th2, sigma_ext, psf_params)
    )

    # Fit H0: point source only.
    fit_h0 = fit_point_only(
        counts=counts,
        point_template=point_template,
        background=background,
    )

    # Fit H1: point source plus fixed-width extension.
    fit_h1 = fit_point_plus_extension(
        counts=counts,
        point_template=point_template,
        ext_template=ext_template,
        background=background,
    )

    nll_min = fit_h1["nll"]
    best_ext_counts = fit_h1["ext_counts"]

    target_nll = nll_min + delta_ts / 2.0

    def profile_function(ext_counts):
        profiled_nll, _ = profile_nll_fixed_extension(
            ext_counts=ext_counts,
            counts=counts,
            point_template=point_template,
            ext_template=ext_template,
            background=background,
        )

        return profiled_nll

    # Find an upper bracket for the root.
    lower = best_ext_counts
    upper = max(2.0 * lower, 1.0)

    for _ in range(100):
        if profile_function(upper) >= target_nll:
            break

        upper *= 2.0
    else:
        raise RuntimeError(
            "Could not bracket the upper-limit solution."
        )

    # Solve:
    #
    # profile_nll(ext_counts) =
    # minimum_nll + delta_ts / 2
    #
    ext_counts_ul = brentq(
        lambda ext_counts:
            profile_function(ext_counts) - target_nll,
        lower,
        upper,
    )

    # Refit the point-source normalization at the upper limit.
    nll_ul, point_counts_at_ul = profile_nll_fixed_extension(
        ext_counts=ext_counts_ul,
        counts=counts,
        point_template=point_template,
        ext_template=ext_template,
        background=background,
    )

    # Two useful ways to express the result.
    point_counts_best = fit_h1["point_counts"]

    ext_to_point_ratio = ext_counts_ul / max(point_counts_best, 1e-12)

    ext_fraction_at_ul = (
        ext_counts_ul /
        max(ext_counts_ul + point_counts_at_ul, 1e-12)
    )

    return {
        "sigma_ext": sigma_ext,

        # Upper limit in expected extended-source counts
        "ext_counts_ul": ext_counts_ul,

        # Best-fit values
        "point_counts_best": point_counts_best,
        "ext_counts_best": best_ext_counts,

        # Point-source normalization after profiling at the UL
        "point_counts_at_ul": point_counts_at_ul,

        # Ratios
        "ext_to_point_ratio": ext_to_point_ratio,
        "ext_fraction_at_ul": ext_fraction_at_ul,

        # Likelihood values
        "nll_h0": fit_h0["nll"],
        "nll_h1": fit_h1["nll"],
        "nll_at_ul": nll_ul,

        # Templates and fitted models
        "point_template": point_template,
        "ext_template": ext_template,
        "fit_h0": fit_h0,
        "fit_h1": fit_h1,
    }


# ------------------------------------------------------------
# Upper limits for multiple energy bins
# ------------------------------------------------------------

def calculate_upper_limits(
    cts,
    th2,
    energies,
    sigmas_ext,
    psf_hbu,
    psf_e_means,
    background=None,
    delta_ts=2.71,
):
    """
    Calculate upper limits for all energy bins and extension sizes.

    Parameters
    ----------
    cts : array, shape (n_energy, n_theta2)
        Counts for each energy bin.
    th2 : array
        theta^2 bin centers.
    energies : array
        Energy-bin centers.
    sigmas_ext : array
        Assumed extension widths.
    psf_hbu : object
        Object containing PSF parameters in psf_hbu.data.
    psf_e_means : array
        Energies associated with the PSF models.
    background : array or None
        Known background. Can have shape:
            (n_theta2,)
        or:
            (n_energy, n_theta2)
    delta_ts : float
        2.71 for an approximate one-sided 95% upper limit.

    Returns
    -------
    results : dict
        Arrays with shape (n_sigma_ext, n_energy).
    """

    cts = np.asarray(cts, dtype=float)
    th2 = np.asarray(th2, dtype=float)
    energies = np.asarray(energies, dtype=float)
    sigmas_ext = np.asarray(sigmas_ext, dtype=float)

    if cts.ndim != 2:
        raise ValueError(
            "cts must have shape (n_energy, n_theta2)."
        )

    n_energy, n_theta2 = cts.shape

    if len(energies) != n_energy:
        raise ValueError(
            "energies must have one value per energy bin."
        )

    if len(th2) != n_theta2:
        raise ValueError(
            "th2 must have one value per theta^2 bin."
        )

    # Prepare output arrays.
    shape = (len(sigmas_ext), n_energy)

    ext_counts_ul = np.zeros(shape)
    ext_to_point_ratio = np.zeros(shape)
    ext_fraction_at_ul = np.zeros(shape)

    best_point_counts = np.zeros(shape)
    best_ext_counts = np.zeros(shape)

    # Loop over energy bins.
    for i, energy in enumerate(energies):

        # Use the PSF closest in energy.
        psf_index = np.argmin(
            np.abs(np.asarray(psf_e_means) - energy)
        )

        psf_params = psf_hbu.data[psf_index][0]

        # Select background for this energy bin.
        if background is None:
            background_i = None

        else:
            background_array = np.asarray(background, dtype=float)

            if background_array.ndim == 1:
                background_i = background_array

            elif background_array.ndim == 2:
                background_i = background_array[i]

            else:
                raise ValueError(
                    "background must be one- or two-dimensional."
                )

        # Loop over assumed extension sizes.
        for k, sigma_ext in enumerate(sigmas_ext):

            result = calculate_upper_limit_one_dataset(
                counts=cts[i],
                th2=th2,
                sigma_ext=sigma_ext,
                psf_params=psf_params,
                background=background_i,
                delta_ts=delta_ts,
            )

            ext_counts_ul[k, i] = result["ext_counts_ul"]
            ext_to_point_ratio[k, i] = result["ext_to_point_ratio"]
            ext_fraction_at_ul[k, i] = result["ext_fraction_at_ul"]

            best_point_counts[k, i] = result["point_counts_best"]
            best_ext_counts[k, i] = result["ext_counts_best"]

    return {
        "ext_counts_ul": ext_counts_ul,
        "ext_to_point_ratio": ext_to_point_ratio,
        "ext_fraction_at_ul": ext_fraction_at_ul,
        "best_point_counts": best_point_counts,
        "best_ext_counts": best_ext_counts,
    }


# ------------------------------------------------------------
# Expected sensitivity from many null simulations
# ------------------------------------------------------------

def calculate_expected_sensitivity(
    cts_toys,
    th2,
    energies,
    sigmas_ext,
    psf_hbu,
    psf_e_means,
    background=None,
    delta_ts=2.71,
):
    """
    Calculate median expected upper limits from many
    no-extension pseudo-experiments.

    Parameters
    ----------
    cts_toys : array, shape
        Either:
            (n_toys, n_energy, n_theta2)
        or:
            (n_energy, n_theta2)
        for one toy.

    Returns
    -------
    sensitivity : dict
        Median and percentile bands.
    """

    cts_toys = np.asarray(cts_toys, dtype=float)

    if cts_toys.ndim == 2:
        cts_toys = cts_toys[None, :, :]

    if cts_toys.ndim != 3:
        raise ValueError(
            "cts_toys must have shape "
            "(n_toys, n_energy, n_theta2)."
        )

    n_toys = cts_toys.shape[0]

    all_ul = []

    for toy_index in range(n_toys):

        result = calculate_upper_limits(
            cts=cts_toys[toy_index],
            th2=th2,
            energies=energies,
            sigmas_ext=sigmas_ext,
            psf_hbu=psf_hbu,
            psf_e_means=psf_e_means,
            background=background,
            delta_ts=delta_ts,
        )

        all_ul.append(result["ext_counts_ul"])

    all_ul = np.asarray(all_ul)

    return {
        "all_upper_limits": all_ul,

        # Median expected upper limit
        "median": np.percentile(all_ul, 50.0, axis=0),

        # Approximate 68% expected band
        "lower_68": np.percentile(all_ul, 16.0, axis=0),
        "upper_68": np.percentile(all_ul, 84.0, axis=0),

        # Approximate 95% expected band
        "lower_95": np.percentile(all_ul, 2.5, axis=0),
        "upper_95": np.percentile(all_ul, 97.5, axis=0),
    }
